In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

train_df = pd.read_csv("data/IMT2024024_train_var1.csv")
test_df  = pd.read_csv("data/IMT2024024_test_var1.csv")

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}")
print("\nFirst 5 rows of Training Data:")
print(train_df.head())

# print("\nMissing values check:")
# print(train_df.isnull().sum())

Train Dataset Shape: (1000, 7)
Test Dataset Shape:  (1000, 6)

First 5 rows of Training Data:
         x1        x2        x3        x4        x5        x6         y
0 -1.000000 -0.514132  0.103341 -0.144700  0.575898  0.785434  0.976873
1 -0.304260 -1.000000 -0.779001 -0.148084 -1.000000 -1.000000  1.621223
2  0.079625 -1.000000 -0.694604  0.068184  0.653448  0.841697 -0.081755
3  1.000000  1.000000  0.268996  0.277195  0.186076  0.714304 -0.456919
4 -0.510508 -0.982768 -0.566677  0.461428  0.689030  0.429986 -0.567492


In [4]:
feature_names = [col for col in train_df.columns if col != 'y']

X_train_raw = train_df[feature_names].to_numpy()
y_train = train_df['y'].to_numpy().reshape(-1, 1)
X_test_raw  = test_df[feature_names].to_numpy()

print(f"X_train shape: {X_train_raw.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test_raw.shape}")

X_train shape: (1000, 6)
y_train shape: (1000, 1)
X_test shape:  (1000, 6)


In [ ]:
# 1. Closed-form Ridge Normal Equation (Lecture 7)
def fit_ridge(X_design, y, lam=1e-3):                   # actually no need of this function
    """
    Solves: w = (X^T @ X + lam * N * I)^(-1) @ X^T @ y
    """
    N, D = X_design.shape
    I = np.eye(D)
    I[0, 0] = 0.0  # Do not regularize the bias/intercept column
    
    # np.linalg.solve solves A w = b directly without computing an explicit inverse
    w = np.linalg.solve(X_design.T @ X_design + lam * N * I, X_design.T @ y)
    return w

# 2. Linear Prediction: y_hat = X @ w
def predict(X_design, w):
    return X_design @ w

# 3. Mean Squared Error (MSE): (1/N) * sum((y - y_hat)^2)
def compute_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# 4. R^2 Score (Coefficient of Determination)
def compute_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (ss_res / ss_tot)

In [ ]:
# TEST we are scaling the features after in kfold validation code

# normalising the features...  -> can use this while writing report

from sklearn.preprocessing import StandardScaler

# 1. Initialize the scaler
scaler = StandardScaler()

# 2. Learn mean and standard deviation from training data and transform it
X_train_scaled = scaler.fit_transform(X_train_raw)

# 3. Transform the test data using the EXACT same training parameters
X_test_scaled = scaler.transform(X_test_raw)

# Inspect the result
print("Original X_train first row:\n", X_train_raw[0])
print("\nScaled X_train first row:\n", X_train_scaled[0])
print(f"\nMean of scaled features: {X_train_scaled.mean(axis=0).round(4)}")
print(f"Std of scaled features:  {X_train_scaled.std(axis=0).round(4)}")

Original X_train first row:
 [-1.         -0.51413194  0.10334119 -0.14470026  0.57589791  0.78543393]

Scaled X_train first row:
 [-1.35732475 -0.67832891  0.15477515 -0.18131852  0.84624156  1.16462628]

Mean of scaled features: [-0. -0. -0. -0.  0. -0.]
Std of scaled features:  [1. 1. 1. 1. 1. 1.]


In [ ]:
# TEST

# expanding our 6 scaled features into a polynomial design matrix   --> maybe i can breif on why i used sklearn why not numpy everywhere and all in the report.

from sklearn.preprocessing import PolynomialFeatures

# 1. Choose a trial degree (let's use degree 2 first)
trial_degree = 2

# 2. Initialize the polynomial transformer
# include_bias=True automatically inserts the leading column of 1s (for w0)
poly = PolynomialFeatures(degree=trial_degree, include_bias=True)

# 3. Transform the scaled training features
X_train_poly = poly.fit_transform(X_train_scaled)

# 4. Transform the scaled test features using the exact same transformer
X_test_poly = poly.transform(X_test_scaled)

# Inspect shapes and names
print(f"Original scaled features shape: {X_train_scaled.shape}")
print(f"Expanded polynomial features shape (degree {trial_degree}): {X_train_poly.shape}")  # total it is 28 features 6+2 C 2 (all possible splits)
print(f"\nTotal terms generated: {X_train_poly.shape[1]}")
print(f"First 10 feature names:\n{poly.get_feature_names_out()[:10]}")

Original scaled features shape: (1000, 6)
Expanded polynomial features shape (degree 2): (1000, 28)

Total terms generated: 28
First 10 feature names:
['1' 'x0' 'x1' 'x2' 'x3' 'x4' 'x5' 'x0^2' 'x0 x1' 'x0 x2' 'x0 x3' 'x0 x4'
 'x0 x5' 'x1^2' 'x1 x2' 'x1 x3' 'x1 x4' 'x1 x5' 'x2^2' 'x2 x3' 'x2 x4'
 'x2 x5' 'x3^2' 'x3 x4' 'x3 x5' 'x4^2' 'x4 x5' 'x5^2']


In [ ]:
# TEST
# 1. Choose a regularization strength (lambda from Lecture 7) --> here ican erite in report why i used L2 what about  L1 and all....
# this and the above block are just examples.... 
lam = 1e-4

# 2. Fit the weights using your NumPy Ridge solver
w = fit_ridge(X_train_poly, y_train, lam=lam)

# 3. Predict on the training data
y_train_pred = predict(X_train_poly, w)

# 4. Compute training performance metrics
train_mse = compute_mse(y_train, y_train_pred)
train_r2 = compute_r2(y_train, y_train_pred)

print(f"Weights vector shape: {w.shape}")
print(f"Bias term (w0):       {w[0, 0]:.4f}")
print(f"Training MSE:         {train_mse:.4e}")
print(f"Training R2 Score:    {train_r2:.4f}")

Weights vector shape: (28, 1)
Bias term (w0):       -0.0597
Training MSE:         2.9822e+00
Training R2 Score:    0.6950


In [ ]:
# here no need for scaling but i did it because- why not!

# now we shall split our training data into train and validation and use K-fold validationa and then find the optimal lambda
# doubt : how will our degree change...??

# also in report i can write why iam using KFold validation instead of simple train-validation split!
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# Convert to NumPy arrays so integer fold indices always select rows correctly
X = np.asarray(X_train_raw)
y = np.asarray(y_train)

degrees = list(range(1, 11))
lambdas = [1e-4, 1e-3, 1e-2, 0.1, 0, 1.0, 10.0]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_results = []
best_val_mse = float("inf")
best_degree = None
best_lambda = None
best_val_r2 = None

print("Running 5-Fold Cross-Validation Grid Search...")
print(f"Degrees: {degrees}")
print(f"Lambdas: {lambdas}")
print("-" * 75)

for d in degrees:
    # Dictionary to collect fold errors for each lambda: {lam: [fold1_mse, fold2_mse, ...]}
    metrics_per_lambda = {lam: {'train_mse': [], 'val_mse': [], 'val_r2': []} for lam in lambdas}           # this is done to optimise the code!

    for train_idx, val_idx in kf.split(X_train_raw):
        # A. Split into training and validation folds
        X_train_fold, X_val_fold = X_train_raw[train_idx], X_train_raw[val_idx]
        y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]

        # B. Standardize features (strictly on the training fold)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_fold)
        X_val_scaled   = scaler.transform(X_val_fold)

        # C. Expand polynomial features
        poly = PolynomialFeatures(degree=d, include_bias=True)
        X_train_poly = poly.fit_transform(X_train_scaled)
        X_val_poly   = poly.transform(X_val_scaled)

        # D. Precompute XT_X and XT_y ONCE for this fold
        N_fold, D_fold = X_train_poly.shape
        XT_X = X_train_poly.T @ X_train_poly
        XT_y = X_train_poly.T @ y_train_fold

        # Identity matrix for regularizing this fold
        I = np.eye(D_fold)
        I[0, 0] = 0.0  # Do not regularize the intercept/bias term

        # E. Quickly test all lambda values on this fold using the precomputed matrices
        for lam in lambdas:
            # Solve (X^T X + lam * N * I) w = X^T y
            weights = np.linalg.solve(XT_X + lam * N_fold * I, XT_y)

            # Evaluate predictions
            y_train_pred = predict(X_train_poly, weights)
            y_val_pred   = predict(X_val_poly, weights)

            metrics_per_lambda[lam]['train_mse'].append(compute_mse(y_train_fold, y_train_pred))
            metrics_per_lambda[lam]['val_mse'].append(compute_mse(y_val_fold, y_val_pred))
            metrics_per_lambda[lam]['val_r2'].append(compute_r2(y_val_fold, y_val_pred))

    # Average results across all 5 folds for each lambda at degree d
    for lam in lambdas:
        mean_tr_mse  = np.mean(metrics_per_lambda[lam]['train_mse'])
        mean_val_mse = np.mean(metrics_per_lambda[lam]['val_mse'])
        mean_val_r2  = np.mean(metrics_per_lambda[lam]['val_r2'])

        grid_results.append({
            'degree': d,
            'lambda': lam,
            'train_mse': mean_tr_mse,
            'val_mse': mean_val_mse,
            'val_r2': mean_val_r2,
            'total_features': D_fold
        })

        if mean_val_mse < best_val_mse:
            best_val_mse = mean_val_mse
            best_degree  = d
            best_lambda  = lam
            best_val_r2  = mean_val_r2

    print(f"Degree {d:2d} finished (tested {len(lambdas)} lambdas across 5 folds).")

# Convert results into a DataFrame for review
results_df = pd.DataFrame(grid_results)

print("\nTop 5 Configurations (Ranked by Validation MSE):")
print(results_df.sort_values(by='val_mse').head(5).to_string(index=False))

print("\n" + "=" * 60)
print(f"Optimal Degree:       {best_degree}")
print(f"Optimal Lambda:       {best_lambda}")
print(f"Best Validation MSE:  {best_val_mse:.4e}")
print(f"Corresponding Val R2: {best_val_r2:.4f}")
print("=" * 60)



Running 5-Fold Cross-Validation Grid Search...
Degrees: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Lambdas: [0.0001, 0.001, 0.01, 0.1, 0, 1.0, 10.0]
---------------------------------------------------------------------------
Degree  1 finished (tested 7 lambdas across 5 folds).
Degree  2 finished (tested 7 lambdas across 5 folds).
Degree  3 finished (tested 7 lambdas across 5 folds).
Degree  4 finished (tested 7 lambdas across 5 folds).
Degree  5 finished (tested 7 lambdas across 5 folds).
Degree  6 finished (tested 7 lambdas across 5 folds).
Degree  7 finished (tested 7 lambdas across 5 folds).
Degree  8 finished (tested 7 lambdas across 5 folds).
Degree  9 finished (tested 7 lambdas across 5 folds).
Degree 10 finished (tested 7 lambdas across 5 folds).

Top 5 Configurations (Ranked by Validation MSE):
 degree  lambda  train_mse  val_mse   val_r2  total_features
      5    0.10   0.246854 0.568600 0.940706             462
      5    0.01   0.148474 0.651928 0.931974             462
      6    0.

In [ ]:
# =====================================================================
# STAGE 2: TRAIN ON 100% OF TRAINING DATA & PREDICT ON TEST
# =====================================================================

# 1. Fit scaler on ALL training samples and transform test data
final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_raw)
X_test_scaled  = final_scaler.transform(X_test_raw)

# 2. Expand polynomial features using the winning degree from K-Fold
final_poly = PolynomialFeatures(degree=best_degree, include_bias=True)
X_train_poly = final_poly.fit_transform(X_train_scaled)
X_test_poly  = final_poly.transform(X_test_scaled)

# 3. Fit final weights using the winning lambda from K-Fold
final_weights = fit_ridge(X_train_poly, y_train, lam=best_lambda)

# 4. Compute predictions on test data
y_test_pred = predict(X_test_poly, final_weights).ravel()

# 5. Create a DataFrame containing ONLY the predicted 'y' column
submission_df = pd.DataFrame({'y': y_test_pred})

csv_filename = "data/IMT2024024_pred_var1.csv"
submission_df.to_csv(csv_filename, index=False)

# excel_filename = "data/IMT2024024_pred_var1.xlsx"
# submission_df.to_excel(excel_filename, index=False)

print(f"Predictions saved with only column 'y' to:")
print(f" - CSV:   {csv_filename}")
# print(f" - Excel: {excel_filename}")
print("\nFirst 5 predicted rows:")
print(submission_df.head())

Predictions saved with only column 'y' to:
 - CSV:   data/IMT2024024_pred_var1.csv

First 5 predicted rows:
          y
0  2.211434
1  3.784487
2  0.415166
3 -4.011903
4 -1.926338


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# Convert to NumPy arrays so indexing behaves consistently
X = np.asarray(X_train_raw)
y = np.asarray(y_train)

# 1. Split data into training and validation sets
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 2. Standardize using only the training split
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_split)
X_val_scaled = scaler.transform(X_val_split)

# 3. Values to test
degrees = list(range(1, 11))
lambdas = [1e-4, 1e-3, 1e-2, 0.1, 0, 1.0, 10.0]

results = []

print("Running Train-validation split Grid Search...")
print(f"Degrees: {degrees}")
print(f"Lambdas: {lambdas}")
print("-" * 75)

# 4. Test every degree-lambda combination
for degree in degrees:
    for lam in lambdas:

        # Generate polynomial features
        poly = PolynomialFeatures(
            degree=degree,
            include_bias=True
        )

        X_train_poly = poly.fit_transform(X_train_scaled)
        X_val_poly = poly.transform(X_val_scaled)

        # Fit Ridge regression
        weights = fit_ridge(
            X_train_poly,
            y_train_split,
            lam=lam
        )

        # Predictions
        y_train_pred = predict(X_train_poly, weights)
        y_val_pred = predict(X_val_poly, weights)

        # Metrics
        train_mse = compute_mse(y_train_split, y_train_pred)
        val_mse = compute_mse(y_val_split, y_val_pred)
        val_r2 = compute_r2(y_val_split, y_val_pred)

        # Store results
        results.append({
            "degree": degree,
            "lambda": lam,
            "train_mse": train_mse,
            "val_mse": val_mse,
            "val_r2": val_r2,
            "total_features": X_train_poly.shape[1]
        })

        # print(
        #     f"{degree:6d} | "
        #     f"{lam:8.1e} | "
        #     f"{train_mse:11.4e} | "
        #     f"{val_mse:11.4e} | "
        #     f"{val_r2:8.4f} | "
        #     f"{X_train_poly.shape[1]:8d}"
        # )

# 5. Convert results to a DataFrame
results_df = pd.DataFrame(results)

# 6. Select the configuration with the lowest validation MSE
best_result = results_df.loc[
    results_df["val_mse"].idxmin()
]

best_degree = int(best_result["degree"])
best_lambda = best_result["lambda"]
best_train_mse = best_result["train_mse"]
best_val_mse = best_result["val_mse"]
best_val_r2 = best_result["val_r2"]
best_total_features = int(best_result["total_features"])

# 7. Display the best configuration
print("\n" + "=" * 75)
print(f"Optimal Degree:       {best_degree}")
print(f"Optimal Lambda:       {best_lambda}")
print(f"Training MSE:         {best_train_mse:.4e}")
print(f"Validation MSE:       {best_val_mse:.4e}")
print(f"Validation R2:        {best_val_r2:.4f}")
print(f"Total Features:       {best_total_features}")
print("=" * 75)

# 8. Display the top configurations
print("\nTop 10 configurations ranked by validation MSE:")
print(
    results_df
    .sort_values(by="val_mse")
    .head(10)
    .to_string(index=False)
)
print("=" * 75)

Degree | Lambda   | Train MSE   | Val MSE     | Val R2   | Features
---------------------------------------------------------------------------

Optimal Degree:       5
Optimal Lambda:       0.01
Training MSE:         1.4391e-01
Validation MSE:       6.8915e-01
Validation R2:        0.9232
Total Features:       462

Top 10 configurations ranked by validation MSE:
 degree  lambda  train_mse  val_mse   val_r2  total_features
      5  0.0100   0.143905 0.689146 0.923220             462
      5  0.1000   0.237708 0.699350 0.922083             462
      6  0.1000   0.172957 0.831037 0.907411             924
      4  0.0100   0.396797 0.889124 0.900940             210
      4  0.0010   0.383306 0.913432 0.898231             210
      4  0.0001   0.382963 0.920503 0.897443             210
      4  0.1000   0.522862 0.930854 0.896290             210
      7  1.0000   0.315647 0.978741 0.890955            1716
      6  1.0000   0.438076 0.983876 0.890383             924
      7  0.1000   0.1137